# ALERTA-TAR Antioquia: Ingesta de Datos (Capa Bronze)
**Gobernación de Antioquia — Secretaría Seccional de Salud y Protección Social**  
*Ingesta estandarizada de Ficha 850 SIVIGILA y eventos longitudinales hacia arquitectura Medallion.*

In [ ]:
import hashlib
import json
from pathlib import Path
import pandas as pd

# =============================================================================
# 1. GESTIÓN DE RUTAS DEL PROYECTO
# =============================================================================
BASE_DIR = Path.cwd()
if BASE_DIR.name == "notebooks":
    BASE_DIR = BASE_DIR.parent

DIR_LANDING = BASE_DIR / "data" / "landing"
DIR_BRONZE = BASE_DIR / "data" / "bronze"

RUTA_SIVIGILA = DIR_LANDING / "sivigila_vih.csv"
RUTA_EVENTOS_ORIGEN = DIR_LANDING / "eventos_longitudinal_vih.csv"

RUTA_BRONZE_COHORTE = DIR_BRONZE / "cohorte_vih_bronze.csv"
RUTA_BRONZE_EVENTOS = DIR_BRONZE / "eventos_longitudinal_vih_bronze.csv"
RUTA_METADATA = DIR_BRONZE / "_metadata_ingesta.json"

DIR_BRONZE.mkdir(parents=True, exist_ok=True)

print("Directorio base configurado:", BASE_DIR)
print("Buscando insumos en:", DIR_LANDING)


In [ ]:
# =============================================================================
# 2. INGESTA Y HOMOLOGACIÓN DE SIVIGILA (COHORTE BASAL)
# =============================================================================
if not RUTA_SIVIGILA.exists():
    raise FileNotFoundError(f"No se encontró el archivo en {RUTA_SIVIGILA}")

print(f"Cargando cohorte SIVIGILA desde: {RUTA_SIVIGILA.name} ...")
df_sivigila = pd.read_csv(
    RUTA_SIVIGILA, sep=";", encoding="latin1", low_memory=False
)
print(
    f"SIVIGILA leído con éxito: {len(df_sivigila):,} filas y {len(df_sivigila.columns)} columnas."
)

mapeo_columnas = {
    "ID": "Llave",
    "edad_": "Edad",
    "sexo_": "Sexo",
    "nmun_resi": "Municipio residencia",
    "ndep_resi": "Departamento",
    "tip_ss_": "Régimen de afiliación",
    "cod_ase_": "EAPB / Aseguradora",
    "est_cli": "Estadío clínico al diagnóstico",
    "fec_not": "Fecha de Notificación",
    "fec_con_": "Fecha de Consulta",
    "pac_hos_": "Hospitalización previa",
    "tub_pul": "Coinfección Tuberculosis",
    "hep_b": "Coinfección Hepatitis B",
    "hep_c": "Coinfección Hepatitis C",
}

df_bronze_cohorte = df_sivigila.rename(columns=mapeo_columnas)


In [ ]:
# =============================================================================
# 3. CONTROL DE TRAZABILIDAD (HASH SHA-256)
# =============================================================================
def calcular_sha256(ruta_archivo: Path) -> str:
    hasher = hashlib.sha256()
    with open(ruta_archivo, "rb") as f:
        while chunk := f.read(65536):
            hasher.update(chunk)
    return hasher.hexdigest()

sha256_sivigila = calcular_sha256(RUTA_SIVIGILA)
print(f"Hash SHA-256 verificado (SIVIGILA): {sha256_sivigila}")


In [ ]:
# =============================================================================
# 4. GUARDADO EN CAPA BRONZE Y GESTIÓN DE EVENTOS LONGITUDINALES
# =============================================================================
df_bronze_cohorte.to_csv(RUTA_BRONZE_COHORTE, index=False, encoding="utf-8-sig")
print(f"Cohorte guardada en Bronze: {RUTA_BRONZE_COHORTE}")

eventos_ingestados = False
sha256_eventos = None
registros_eventos = 0

if RUTA_EVENTOS_ORIGEN.exists():
    print(f"Cargando eventos longitudinales desde: {RUTA_EVENTOS_ORIGEN.name} ...")
    df_eventos = pd.read_csv(RUTA_EVENTOS_ORIGEN, low_memory=False)
    df_eventos.to_csv(RUTA_BRONZE_EVENTOS, index=False, encoding="utf-8-sig")
    sha256_eventos = calcular_sha256(RUTA_EVENTOS_ORIGEN)
    eventos_ingestados = True
    registros_eventos = len(df_eventos)
    print(f"Eventos guardados en Bronze ({registros_eventos:,} registros).")
else:
    print("Aviso: 'eventos_longitudinal_vih.csv' no encontrado en landing. Se conserva el flujo sin eventos auxiliares.")


In [ ]:
# =============================================================================
# 5. EXPORTACIÓN DE METADATOS DE AUDITORÍA
# =============================================================================
metadata = {
    "proyecto": "ALERTA-TAR Antioquia",
    "archivo_origen": RUTA_SIVIGILA.name,
    "sha256": sha256_sivigila,
    "total_registros": int(len(df_bronze_cohorte)),
    "pacientes_unicos": int(df_bronze_cohorte["Llave"].nunique()),
    "columnas": int(len(df_bronze_cohorte.columns)),
    "tipo_datos": "Reales - SIVIGILA Ficha 850 INS",
    "eventos_longitudinales": {
        "incluido": eventos_ingestados,
        "registros": registros_eventos,
        "sha256": sha256_eventos,
    },
}

with open(RUTA_METADATA, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=4)

print("Metadatos de auditoría exportados en:", RUTA_METADATA.name)
print("\n¡Ejecución de Bronze completada exitosamente!")
